# PANDORA-big5 — Google Colab Experiment Runner

This notebook is the **Django-free experimental harness** for the `pandora` branch of
`Popthemy/personality-prediction-app`.

## Current experiment definition

**2 × 2 × 2 factorial = 8 conditions**

| Factor | Levels |
|---|---|
| Comment selection | Baseline vs **Q-learning** |
| Augmentation | No GAN vs **paired (embedding, OCEAN) GAN** |
| Model | **Lasso/ElasticNet** vs **LSTM**, both continuous 5-output OCEAN regression |

Pipeline:

`PANDORA → cleaning/grouping → participant split → baseline/Q-learning selection → BERT (768-D) → optional paired GAN augmentation on training fold → Lasso or LSTM → continuous OCEAN predictions → common metrics → High/Low threshold + ROC/PR analysis`

### Important experimental rules

- **20 users** is the initial smoke-test size.
- The train/validation/test split is **participant-level**.
- BERT is `bert-base-uncased`; the project uses the **[CLS] 768-D representation**.
- GAN augmentation is trained on **training participants only** and generates paired embeddings + OCEAN targets.
- Lasso and LSTM both predict **5 continuous OCEAN values**. LSTM is **not** a 3-class classifier in the current branch.
- Binary High/Low analysis is downstream from continuous predictions.
- Test data is not used to choose the decision threshold.
- Expensive data, embeddings, artifacts, and experiment state are stored on **Google Drive**, not SharePoint.

**Start with the cells below. Do not use the old notebook generated from the pre-update architecture.**


## 1. Google Drive persistence

In [ ]:

from pathlib import Path
import json, os, shutil, subprocess, sys, time

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/pandora_personality")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./pandora_personality").resolve()
    print("Not running inside Colab; using:", DRIVE_ROOT)

DATA_DIR       = DRIVE_ROOT / "data"
CACHE_DIR      = DRIVE_ROOT / "cache"
ARTIFACT_DIR   = DRIVE_ROOT / "artifacts"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
STATE_DIR      = DRIVE_ROOT / "state"
REPO_CACHE_DIR = DRIVE_ROOT / "repo_cache"

for d in [DATA_DIR, CACHE_DIR, ARTIFACT_DIR, CHECKPOINT_DIR, STATE_DIR, REPO_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STATE_FILE = STATE_DIR / "colab_state.json"
CONFIG_FILE = STATE_DIR / "experiment_config.json"

print("Persistent Drive root:", DRIVE_ROOT)
print("Data:", DATA_DIR)
print("BERT cache:", CACHE_DIR)
print("Artifacts:", ARTIFACT_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("State:", STATE_DIR)


## 2. Runtime / GPU check

In [ ]:

import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available. For the smoke test, use Runtime → Change runtime type → GPU.")


## 3. Get the exact project code

The notebook uses the **`pandora` branch** directly. The runtime copy is disposable; all experiment
data and artifacts remain on Drive.

The commit SHA is recorded in Drive so the results can be tied to the exact code version used.


In [ ]:

REPO_URL = "https://github.com/Popthemy/personality-prediction-app.git"
REPO_BRANCH = "pandora"
REPO_DIR = Path("/content/personality-prediction-app")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
(REPO_CACHE_DIR / "repo_commit.txt").write_text(commit, encoding="utf-8")

print("Repo:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Commit:", commit)
print("backend exists:", (REPO_DIR / "backend").is_dir())


## 4. Install the Colab dependency set

The repository includes `requirements-colab.txt`. It deliberately avoids Django/Celery/Redis and
does **not hard-pin PyTorch**, because Colab's CUDA-matched PyTorch build should be preserved.


In [ ]:

REQ_FILE = REPO_DIR / "requirements-colab.txt"
assert REQ_FILE.exists(), f"Missing {REQ_FILE}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)],
    check=True,
)

print("Installed:", REQ_FILE)


## 5. Verify the required project interfaces

In [ ]:

# Import checks catch stale/mismatched notebook ↔ repository contracts before expensive BERT work.
from backend.ml_pipeline.services.data.pandora import (
    load_pandora_comments, PreparedUserComments, UserTraits
)
from backend.ml_pipeline.services.bert_encoder import BERTEncoder
from backend.ml_pipeline.services.gan_augmenter import GANAugmenter
from backend.ml_pipeline.services.lasso_regressor import LassoTrainer
from backend.ml_pipeline.services.lstm_classifier import LSTMTrainer
from backend.ml_pipeline.services import metrics_engine
from backend.ml_pipeline.experiments.pandora_runner import (
    ExperimentConfig, ExperimentRunner, EXPERIMENTS, OCEAN_TRAITS
)

print("✓ PANDORA ingestion")
print("✓ BERTEncoder")
print("✓ joint GAN augmenter")
print("✓ Lasso trainer")
print("✓ continuous LSTM trainer")
print("✓ metrics engine")
print("✓ factorial ExperimentRunner")
print("Conditions:", list(EXPERIMENTS))


## 6. Download PANDORA-big5 to Drive

The dataset is downloaded from the Hugging Face dataset repository and stored under `DATA_DIR`.
The download is idempotent: if the file is already present in Drive, it is reused.

The current parquet export does not expose Reddit usernames. The project therefore groups rows by
the `(O,C,E,A,N)` trait tuple as a **proxy-user key**. This is acceptable for the pipeline smoke test;
it must remain explicitly documented when interpreting final research results.


In [ ]:

from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "jingjietan/pandora-big5"

repo_files = list_repo_files(HF_REPO, repo_type="dataset")
parquet_files = [f for f in repo_files if f.endswith(".parquet") and f.startswith("data/")]
if not parquet_files:
    parquet_files = [f for f in repo_files if f.endswith(".parquet")]

if not parquet_files:
    raise RuntimeError(f"No parquet file found in {HF_REPO}")

local_parquets = []
for rel_path in parquet_files:
    target = DATA_DIR / rel_path
    if target.exists():
        p = target
        print("Reusing:", p)
    else:
        p = Path(hf_hub_download(
            HF_REPO, rel_path, repo_type="dataset", local_dir=str(DATA_DIR)
        ))
        print("Downloaded:", p)
    local_parquets.append(p)

PANDORA_FILE = local_parquets[0]
print("Using:", PANDORA_FILE)


## 7. Dataset sanity check / label-scale audit

In [ ]:

import pandas as pd
import numpy as np

required_cols = ["O", "C", "E", "A", "N", "ptype", "text"]
raw_preview = pd.read_parquet(PANDORA_FILE, columns=required_cols)

print("Rows in selected parquet:", len(raw_preview))
print("Columns:", list(raw_preview.columns))
print("\nMissing values:")
display(raw_preview[required_cols].isna().sum().to_frame("missing"))

print("\nRaw OCEAN scale:")
display(raw_preview[["O","C","E","A","N"]].describe().T[["min","max","mean","std"]])

print("\nSample text:")
print(str(raw_preview["text"].dropna().iloc[0])[:500])


## 8. Prepare and persist cleaned PANDORA data

In [ ]:

PREPARED_JSON = DATA_DIR / "pandora_prepared.json"

def _load_prepared_cache(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    prepared = []
    for u in payload:
        traits = UserTraits(**u["traits"]) if u.get("traits") else None
        # CleanedContent is a dataclass in the repo. Reconstruct it by field names.
        from backend.ml_pipeline.cleaning.cleaner import CleanedContent
        comments = [CleanedContent(**c) for c in u.get("comments", [])]
        prepared.append(
            PreparedUserComments(
                user_id=u["user_id"],
                traits=traits,
                comments=comments,
            )
        )
    return prepared

if PREPARED_JSON.exists():
    prepared = _load_prepared_cache(PREPARED_JSON)
    print("✓ Reused prepared data:", PREPARED_JSON)
else:
    prepared = load_pandora_comments(
        str(PANDORA_FILE),
        output_path=str(PREPARED_JSON),
        min_text_length=3,
        group_by="traits",
    )
    print("✓ Created prepared data:", PREPARED_JSON)

eligible = [
    u for u in prepared
    if u.traits is not None and len(u.comments) >= 5
]

print("Prepared proxy-users:", len(prepared))
print("Eligible for smoke test:", len(eligible))
print("Usable comments:", sum(len(u.comments) for u in prepared))

if len(eligible) < 20:
    raise RuntimeError(
        f"Only {len(eligible)} eligible users are available; the planned smoke test needs 20."
    )


## 9. Experiment configuration

These settings are deliberately small for the first end-to-end run. Once the smoke test succeeds,
the same notebook can be rerun with a larger `sample_n_users`.

**Do not change the experimental factors here.** Change only compute-budget parameters when needed.


In [ ]:

cfg = ExperimentConfig(
    # Smoke-test population
    sample_n_users=20,
    min_comments_per_user=5,
    seed=42,

    # Selection
    top_k=10,
    qlearning_train_epochs=3,

    # Participant split
    val_ratio=0.20,
    test_ratio=0.20,

    # Paired GAN
    synthetic_weight=0.35,
    gan_latent_dim=64,
    gan_hidden_dim=128,
    gan_epochs=150,
    gan_batch_size=16,
    gan_learning_rate=2e-4,

    # BERT
    bert_max_length=256,

    # Lasso / ElasticNet
    lasso_alpha=0.001,
    lasso_l1_ratio=0.5,
    lasso_max_iter=10000,
    lasso_regularization="elasticnet",

    # Continuous 5-output LSTM
    lstm_epochs=35,
    lstm_batch_size=4,
    lstm_hidden_dim=128,
    lstm_num_layers=2,
    lstm_dropout=0.2,
    lstm_learning_rate=1e-3,

    # Persistent outputs
    output_dir=str(ARTIFACT_DIR),
    embedding_cache_dir=str(CACHE_DIR),
)

CONFIG_FILE.write_text(json.dumps(cfg.__dict__, indent=2), encoding="utf-8")
print(json.dumps(cfg.__dict__, indent=2))


## 10. Pre-flight validation

This cell checks the configuration and the core research contract before BERT/GAN/LSTM training.


In [ ]:

assert cfg.sample_n_users == 20, "Smoke-test run should use 20 users."
assert cfg.top_k > 0
assert 0 < cfg.val_ratio < 1
assert 0 < cfg.test_ratio < 1
assert cfg.val_ratio + cfg.test_ratio < 1
assert cfg.bert_max_length > 0
assert cfg.lstm_epochs > 0
assert cfg.gan_epochs > 0
assert cfg.output_dir == str(ARTIFACT_DIR)
assert cfg.embedding_cache_dir == str(CACHE_DIR)

print("✓ 20-user smoke test")
print("✓ participant-level split configured")
print("✓ BERT cache on Drive")
print("✓ artifacts on Drive")
print("✓ Lasso + LSTM continuous OCEAN")
print("✓ paired GAN enabled only in GAN conditions")
print("✓ Q-learning and baseline use the same top_k budget")
print("✓ 8 factorial conditions:", len(EXPERIMENTS))


## 11. Run the full 2×2×2 experiment

`ExperimentRunner` is the Django-free orchestration layer for this notebook. It uses the project's
actual ML service classes and keeps the held-out participants separate from training.

**This is the expensive cell.** BERT embeddings are cached to Drive, so a later runtime can reuse
previously encoded text.


In [ ]:

import logging, traceback, time

logging.getLogger("ml_pipeline").setLevel(logging.INFO)

runner = ExperimentRunner(prepared, cfg)

run_started = time.time()
try:
    bundle = runner.run()
except Exception:
    traceback.print_exc()
    raise
finally:
    elapsed = time.time() - run_started
    print(f"Elapsed: {elapsed/60:.1f} minutes")

print("Experiment completed.")
print("Bundle keys:", sorted(bundle.keys()))


## 12. Save a durable completion record

This records the exact repo commit, configuration, and completion time on Drive. It is separate from
the model/artifact files and is useful when a Colab runtime is lost.


In [ ]:

from datetime import datetime, timezone

completion = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL,
    "branch": REPO_BRANCH,
    "commit": commit,
    "pandora_file": str(PANDORA_FILE),
    "prepared_file": str(PREPARED_JSON),
    "config_file": str(CONFIG_FILE),
    "artifact_dir": str(ARTIFACT_DIR),
    "conditions": list(EXPERIMENTS.keys()),
}

STATE_FILE.write_text(json.dumps(completion, indent=2), encoding="utf-8")
print(json.dumps(completion, indent=2))


## 13. Headline results

In [ ]:

print("=== FINDINGS ===")
for note in bundle.get("findings", {}).get("notes", []):
    print("-", note)

for key in ["model_comparison", "factor_effects"]:
    print("\n---", key, "---")
    obj = bundle.get(key)
    if isinstance(obj, dict):
        for subkey, value in obj.items():
            print("\n", subkey)
            try:
                display(value)
            except Exception:
                print(value)
    else:
        try:
            display(obj)
        except Exception:
            print(obj)


## 14. Inspect the canonical metrics output

The continuous regression metrics are the primary evaluation. Binary High/Low thresholding,
ROC-AUC and PR-AUC are downstream analyses.

Do not interpret a binary threshold result as replacing the continuous OCEAN regression result.


In [ ]:

# Show any tabular result objects returned by the runner.
for key, value in bundle.items():
    if isinstance(value, pd.DataFrame):
        print(f"\n=== {key} ===")
        display(value)


## 15. Inspect files persisted to Google Drive

In [ ]:

print("=== ARTIFACTS ===")
for p in sorted(ARTIFACT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(DRIVE_ROOT)}  |  {p.stat().st_size/1024:.1f} KB")

print("\n=== STATE ===")
print(STATE_FILE.read_text(encoding="utf-8") if STATE_FILE.exists() else "No completion state yet.")


## 16. Resume / reconnect checklist

When Colab disconnects:

1. Open this notebook again.
2. Reconnect Google Drive.
3. Rerun cells 1–10.
4. The PANDORA parquet, prepared JSON, BERT cache, artifacts, and state remain on Drive.
5. If the previous **experiment run itself** was interrupted before it finished, rerun the experiment
   cell; do not assume an in-memory `bundle` survived the runtime reset.
6. Use the persisted artifacts/state to determine what completed.

The expensive BERT encoding is specifically cached by the project runner, which is the main protection
against restarting all embedding work after a runtime reset.


## 17. Scale-up configuration

Only after the 20-user smoke test completes successfully should you increase the sample size.

Suggested progression:

`20 → 40 → 60 → final research sample`

Keep the seed and split policy controlled so comparisons remain reproducible.

For the final research run, preserve:
- exact Git commit,
- exact configuration,
- sample size,
- label scale,
- participant split,
- all 8 condition results,
- continuous metrics,
- threshold analysis,
- ROC/PR outputs.


## 18. Research interpretation guardrails

### Primary
- MAE, RMSE and R² for the continuous 5-dimensional OCEAN predictions.
- Compare Lasso and LSTM under the **same selection × GAN condition**.

### Factor effects
- Q-learning effect = Q-learning selection − baseline selection, holding model and GAN fixed.
- GAN effect = GAN − no-GAN, holding model and selection fixed.

### Downstream decision analysis
- Convert continuous scores to **High/Low** using the project's threshold analysis.
- ROC and PR use continuous prediction scores.
- Threshold selection must not be optimized on the final test participants.

### Dataset identity limitation
The downloaded PANDORA parquet lacks literal Reddit usernames. The current ingestion layer therefore
uses the `(O,C,E,A,N)` tuple as a proxy-user key. This is a known dataset limitation and should be
reported rather than presented as true author identity.
